# Local MONAI U-Net MRI Reconstruction Demo

This notebook is a small, inference-only adaptation of the MONAI U-Net MRI reconstruction tutorial for one local fastMRI brain multi-coil sample. It uses your local `.h5` files, loads a pretrained checkpoint if present, runs slice-wise reconstruction, and saves simple local outputs.

## 1. Environment Check

This notebook does not install packages automatically. It checks for the required local Python packages first and stops with a clear message if anything is missing.

In [ ]:
from importlib import import_module

required_modules = {
    "torch": "torch",
    "monai": "monai",
    "h5py": "h5py",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
}

missing = []
for package_name, module_name in required_modules.items():
    try:
        import_module(module_name)
    except Exception as exc:
        missing.append(f"- {package_name}: {exc}")

if missing:
    raise ImportError(
        "Missing required packages for this notebook:\n"
        + "\n".join(missing)
        + "\nInstall them in your local environment before running the next cells."
    )

print("All required packages are importable.")

## 2. Imports

Only the packages needed for local inference, visualization, and saving outputs are imported.

In [ ]:
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch

from monai.apps.reconstruction.complex_utils import complex_abs, convert_to_tensor_complex
from monai.apps.reconstruction.fastmri_reader import FastMRIReader
from monai.apps.reconstruction.mri_utils import root_sum_of_squares
from monai.data.fft_utils import ifftn_centered
from monai.networks.nets import BasicUNet

torch.manual_seed(0)
np.random.seed(0)
plt.rcParams["image.cmap"] = "gray"

print(f"torch: {torch.__version__}")
print(f"monai: {__import__('monai').__version__}")
print(f"numpy: {np.__version__}")
print(f"h5py: {h5py.__version__}")

## 3. Local Path Configuration

Paths are resolved relative to the current repo root. The notebook deterministically picks the first sorted `.h5` file unless you set `manual_sample_name`.

In [ ]:
REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data"
CHECKPOINT_PATH = REPO_ROOT / "demo_checkpoint" / "unet_mri_reconstruction.pt"
OUTPUT_ROOT = REPO_ROOT / "outputs" / "reconstruction_demo"

# Optional manual override, for example: "file_brain_AXFLAIR_210_6001516.h5"
manual_sample_name = None

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Data directory not found: {DATA_DIR}")

sample_files = sorted(DATA_DIR.glob("*.h5"))
if not sample_files:
    raise FileNotFoundError(f"No .h5 files found under {DATA_DIR}")

if manual_sample_name is None:
    sample_file = sample_files[0]
else:
    sample_file = DATA_DIR / manual_sample_name
    if not sample_file.is_file():
        raise FileNotFoundError(f"Requested sample file not found: {sample_file}")

sample_output_dir = OUTPUT_ROOT / sample_file.stem
sample_output_dir.mkdir(parents=True, exist_ok=True)

print(f"Repo root: {REPO_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Checkpoint path: {CHECKPOINT_PATH}")
print(f"Output directory: {sample_output_dir}")
print(f"Found {len(sample_files)} local .h5 files.")
print(f"Selected sample: {sample_file.name}")

## 4. Device Selection

On Apple Silicon, this notebook prefers `mps` when available. Otherwise it falls back to CPU.

In [ ]:
has_mps_backend = hasattr(torch.backends, "mps")
mps_is_built = has_mps_backend and torch.backends.mps.is_built()
mps_is_available = has_mps_backend and torch.backends.mps.is_available()

if mps_is_built and mps_is_available:
    device = torch.device("mps")
    device_note = "Using Apple Silicon MPS backend."
else:
    device = torch.device("cpu")
    if mps_is_built and not mps_is_available:
        device_note = "MPS backend is built but unavailable in this session, so CPU is used."
    else:
        device_note = "MPS backend is not available, so CPU is used."

print(f"Selected device: {device}")
print(device_note)

## 5. Model Loading

Expected checkpoint file:

- File name: `unet_mri_reconstruction.pt`
- Local folder: `demo_checkpoint/`
- Expected full path: `demo_checkpoint/unet_mri_reconstruction.pt`
- Official MONAI checkpoint URL: `https://github.com/Project-MONAI/MONAI-extra-test-data/releases/download/0.8.1/unet_mri_reconstruction.pt`

If the file is missing locally, the next cell raises a clear error instead of inventing a checkpoint.

In [ ]:
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        f"Checkpoint not found: {CHECKPOINT_PATH}\n"
        "Download 'unet_mri_reconstruction.pt' and place it inside 'demo_checkpoint/' before running this cell."
    )

model = BasicUNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=1,
    features=[32, 64, 128, 256, 512, 32],
).to(device)

try:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True)
except TypeError:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)

state_dict = checkpoint["state_dict"] if isinstance(checkpoint, dict) and "state_dict" in checkpoint else checkpoint
model.load_state_dict(state_dict)
model.eval()

print(f"Loaded checkpoint: {CHECKPOINT_PATH.name}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 6. Sample File Discovery And Raw HDF5 Inspection

This cell shows what is physically stored in the selected `.h5` file before MONAI-specific loading.

In [ ]:
def _python_value(value):
    if isinstance(value, bytes):
        return value.decode("utf-8")
    if hasattr(value, "tolist"):
        return value.tolist()
    return value

with h5py.File(sample_file, "r") as h5_file:
    dataset_info = {
        key: {"shape": tuple(h5_file[key].shape), "dtype": str(h5_file[key].dtype)}
        for key in h5_file.keys()
    }
    attrs_info = {key: _python_value(h5_file.attrs[key]) for key in h5_file.attrs.keys()}

print(f"Selected sample file: {sample_file.name}")
print("Datasets:")
for key, info in dataset_info.items():
    print(f"- {key}: shape={info['shape']}, dtype={info['dtype']}")

print("\nAttributes:")
for key, value in attrs_info.items():
    print(f"- {key}: {value}")

## 7. Data Loading And Preprocessing

The local files already contain masked test-style k-space and a `mask`, but they do not contain `reconstruction_rss`. Because of that, this notebook skips the tutorial's dataset split logic and synthetic masking path, and instead builds a zero-filled baseline directly from the loaded masked k-space.

In [ ]:
reader = FastMRIReader()
raw_data = reader.read(str(sample_file))
kspace_np, meta = reader.get_data(raw_data)
mask_np = np.asarray(meta.get("mask")) if "mask" in meta else None
target_np = np.asarray(raw_data["reconstruction_rss"]) if "reconstruction_rss" in raw_data else None

print(f"Reader k-space shape: {kspace_np.shape}")
print(f"Reader k-space dtype: {kspace_np.dtype}")
print(f"Reader mask shape: {None if mask_np is None else mask_np.shape}")
print(f"Reader acquisition: {meta.get('acquisition', 'unknown')}")
print(f"Ground-truth reconstruction available: {target_np is not None}")

if np.iscomplexobj(kspace_np):
    kspace_tensor = convert_to_tensor_complex(kspace_np, dtype=torch.float32)
elif kspace_np.shape[-1] == 2:
    kspace_tensor = torch.as_tensor(kspace_np, dtype=torch.float32)
else:
    raise ValueError(
        f"Unsupported k-space representation: shape={kspace_np.shape}, dtype={kspace_np.dtype}. "
        "Expected complex-valued data or a final size-2 real/imag dimension."
    )

zero_filled_complex = ifftn_centered(kspace_tensor, spatial_dims=2, is_complex=True)
zero_filled_abs = complex_abs(zero_filled_complex)
zero_filled = root_sum_of_squares(zero_filled_abs, spatial_dim=1).to(torch.float32)

# Match the MONAI tutorial input behavior closely: per-slice normalization, then clip.
slice_means = zero_filled.mean(dim=(1, 2), keepdim=True)
slice_stds = zero_filled.std(dim=(1, 2), keepdim=True).clamp_min(1e-8)
zero_filled_norm = ((zero_filled - slice_means) / slice_stds).clamp(-6.0, 6.0)

print(f"Zero-filled volume shape: {tuple(zero_filled.shape)}")
print(f"Normalized input shape: {tuple(zero_filled_norm.shape)}")
print(f"Number of slices: {zero_filled_norm.shape[0]}")

## 8. Inference

The pretrained U-Net is applied slice by slice. Each output slice is denormalized back to the zero-filled intensity scale.

In [ ]:
reconstructed_slices = []

with torch.no_grad():
    for slice_idx in range(zero_filled_norm.shape[0]):
        model_input = zero_filled_norm[slice_idx].unsqueeze(0).unsqueeze(0).to(device)
        model_output = model(model_input).detach().cpu().squeeze(0).squeeze(0)

        slice_mean = slice_means[slice_idx].item()
        slice_std = slice_stds[slice_idx].item()
        model_output = model_output * slice_std + slice_mean
        reconstructed_slices.append(model_output.numpy())

reconstruction_np = np.stack(reconstructed_slices, axis=0)
zero_filled_np = zero_filled.cpu().numpy()

print(f"Reconstruction volume shape: {reconstruction_np.shape}")
print(f"Reconstruction dtype: {reconstruction_np.dtype}")

## 9. Visualization

This section makes the reconstruction easier to read: it prints simple volume-level change statistics and shows three representative slices. The current default view is a 3-row summary grid: zero-filled baseline, U-Net reconstruction, and absolute change.

In [ ]:
change_map_np = np.abs(reconstruction_np - zero_filled_np)
middle_slice_idx = reconstruction_np.shape[0] // 2
preview_slice_indices = sorted({
    max(0, reconstruction_np.shape[0] // 4),
    middle_slice_idx,
    min(reconstruction_np.shape[0] - 1, (3 * reconstruction_np.shape[0]) // 4),
})

window_source = np.concatenate(
    [zero_filled_np.ravel(), reconstruction_np.ravel()] if target_np is None else [zero_filled_np.ravel(), reconstruction_np.ravel(), target_np.ravel()]
)
vmin = float(np.percentile(window_source, 1.0))
vmax = float(np.percentile(window_source, 99.5))
change_vmax = float(np.percentile(change_map_np, 99.5))

print(f"Representative slice indices: {preview_slice_indices}")
print(f"Mean absolute change vs zero-filled: {change_map_np.mean():.6f}")
print(f"Max absolute change vs zero-filled: {change_map_np.max():.6f}")

if target_np is not None:
    target_mae = float(np.mean(np.abs(reconstruction_np - target_np)))
    zero_filled_mae = float(np.mean(np.abs(zero_filled_np - target_np)))
    print(f"Mean absolute error to target, zero-filled: {zero_filled_mae:.6f}")
    print(f"Mean absolute error to target, U-Net: {target_mae:.6f}")

def build_preview_figure(sample_name, zero_filled_volume, reconstruction_volume, change_volume, slice_indices, vmin, vmax, change_vmax):
    fig, axes = plt.subplots(3, len(slice_indices), figsize=(4.2 * len(slice_indices), 10))
    axes = np.atleast_2d(axes)

    row_titles = ["Zero-filled input", "U-Net reconstruction", "Absolute change"]
    row_volumes = [zero_filled_volume, reconstruction_volume, change_volume]
    row_cmaps = ["gray", "gray", "magma"]
    row_vmins = [vmin, vmin, 0.0]
    row_vmaxs = [vmax, vmax, change_vmax]

    for row_idx, (title, volume, cmap, row_vmin, row_vmax) in enumerate(zip(row_titles, row_volumes, row_cmaps, row_vmins, row_vmaxs)):
        for col_idx, slice_idx in enumerate(slice_indices):
            ax = axes[row_idx, col_idx]
            ax.imshow(volume[slice_idx], cmap=cmap, vmin=row_vmin, vmax=row_vmax)
            if row_idx == 0:
                ax.set_title(f"Slice {slice_idx}")
            if col_idx == 0:
                ax.set_ylabel(title)
            ax.axis("off")

    fig.suptitle(f"{sample_name} | local MRI reconstruction summary", fontsize=14)
    fig.tight_layout()
    return fig

preview_fig = build_preview_figure(
    sample_file.name,
    zero_filled_np,
    reconstruction_np,
    change_map_np,
    preview_slice_indices,
    vmin,
    vmax,
    change_vmax,
)

preview_fig

## 10. Save Outputs

The reconstruction, zero-filled baseline, and preview image are saved under `outputs/reconstruction_demo/<sample_stem>/`.

In [ ]:
reconstruction_path = sample_output_dir / "reconstruction.npy"
zero_filled_path = sample_output_dir / "zero_filled.npy"
preview_path = sample_output_dir / "preview.png"

np.save(reconstruction_path, reconstruction_np)
np.save(zero_filled_path, zero_filled_np)

if target_np is not None:
    target_path = sample_output_dir / "target.npy"
    np.save(target_path, target_np)
    print(f"Saved target volume: {target_path}")

preview_fig.savefig(preview_path, dpi=150, bbox_inches="tight")

print(f"Saved reconstruction volume: {reconstruction_path}")
print(f"Saved zero-filled volume: {zero_filled_path}")
print(f"Saved preview image: {preview_path}")

## 11. Short Notes / Limitations

- The current local files are `AXT1` and `AXFLAIR`, while the MONAI U-Net checkpoint was documented on the brain T2 subset, so output quality is only a practical demo here.
- The current local files are test-style masked inputs without `reconstruction_rss`, so this notebook does not compute reconstruction metrics.
- Model-quality scores such as SSIM or PSNR require a target reconstruction for comparison. With the current test-style files, you can inspect the reconstruction visually and summarize how much it changed the zero-filled baseline, but you cannot compute the official benchmark score.
- To run another local sample, set `manual_sample_name` near the top and re-run from the path configuration cell.